# Air Quality Patterns in African Cities
### Data Analytics Capstone — Group 5

**Client:** United Nations Environment Programme (UNEP)
**Team:** Samuel Tokoye, Paul Kibet Miningwa, Winnie Odoyo, Gloria Simiyu Wandabwa
**Data source:** OpenAQ v3 API
**Cities studied:** Nairobi, Kampala, Kigali, Addis Ababa, Johannesburg, Lagos
**Study period:** 2021–present

## Project brief

Air pollution is an increasing public-health concern across African cities. UNEP has commissioned
this analysis to evaluate air-quality patterns across six selected cities, in order to guide
future intervention priorities.

## Research questions

1. Which city recorded the highest average PM2.5 concentration during the study period?
2. Which pollutants are monitored most frequently across the selected cities?
3. Which cities experience the greatest seasonal variation in PM2.5 concentrations?
4. How frequently do PM2.5 measurements exceed the WHO recommended guideline values?
5. Which monitoring stations have the most complete and reliable datasets?
6. Which cities should be prioritised for air-quality intervention programmes, and why?

Every recommendation in this notebook is directly supported by evidence obtained during the
analysis below.


## 1. Data Acquisition: API Documentation

**API:** OpenAQ v3 (`https://api.openaq.org/v3`)
**Authentication:** API key required, sent via the `X-API-Key` header

### Endpoints used

**`GET /v3/locations`** — identify monitoring stations within each city.

| Parameter | Value used | Notes |
|---|---|---|
| `coordinates` | `{lat},{lon}` per city center | e.g. Nairobi: `-1.286389,36.817223` |
| `radius` | `25000` (metres) | 25km — the maximum radius v3 allows |
| `limit` | `100` | Sufficient for all cities studied |

Each location response includes an embedded `sensors` array (pollutant type, sensor ID) and
`datetimeFirst`/`datetimeLast` reporting window, so a separate `/sensors` call was unnecessary.

**`GET /v3/sensors/{sensor_id}/days`** — pull daily-averaged PM2.5 measurements for each active sensor.

| Parameter | Value used | Notes |
|---|---|---|
| `date_from` | `2021-01-01` | Start of study window |
| `date_to` | current date | End of study window |
| `limit` | `1000` | Max page size |
| `page` | incremented until a page returns <1000 results | `meta.found` returned inconsistent formats across responses, so page-size was used as the reliable pagination stop condition instead |

### Preprocessing rules applied during acquisition

- **Pollutant scope:** full time-series pulled for **PM2.5 only**. Other pollutants were catalogued
  by station frequency (for Q2) but not pulled as time-series — none of the six client questions
  require trend data for pollutants other than PM2.5.
- **"Active sensor" definition:** a sensor counts as active if its station's `datetimeLast` falls
  within 2 years of the pull date. Only active sensors had measurement history pulled.
- **Known limitation — Johannesburg:** 7 PM2.5 sensors found, only 1 active. Johannesburg's overall
  monitoring is also skewed toward SO2/PM10/CO rather than PM2.5, likely reflecting
  industrial/regulatory monitoring infrastructure rather than the low-cost sensor networks
  driving PM2.5 density in Kampala, Lagos, and Nairobi. Retained in the study with this limitation
  documented rather than dropped.

### Files produced by acquisition (see `scripts/`)

| File | Contents |
|---|---|
| `data/raw/{city}_locations.json` | Raw station metadata per city |
| `data/raw/pm25_sensors_summary.json` | Extracted PM2.5 sensors, active/inactive flag |
| `data/raw/{city}_measurements.json` | Daily-averaged PM2.5 readings per active sensor |
| `data/raw/pollutant_frequency.csv` | Station counts per pollutant type per city |


## 2. Data Loading

Load the outputs of the acquisition scripts (see `scripts/`) so the rest of this notebook works from saved data, not live API calls.

In [ ]:
import json
import os
import pandas as pd

CITIES = ["nairobi", "kampala", "kigali", "addis_ababa", "johannesburg", "lagos"]
DATA_DIR = "../data/raw" 


### 2.1 Station metadata per city

In [ ]:
locations = {}
for city in CITIES:
    with open(f"{DATA_DIR}/{city}_locations.json") as f:
        locations[city] = json.load(f)
    print(f"{city}: {locations[city]['meta']['found']} stations found")


### 2.2 PM2.5 sensor summary (active/inactive flag)

In [ ]:
with open(f"{DATA_DIR}/pm25_sensors_summary.json") as f:
    sensors_df = pd.DataFrame(json.load(f))

print(f"Total PM2.5 sensors: {len(sensors_df)}")
print(f"Active sensors: {sensors_df['is_active'].sum()}")
sensors_df.groupby("city")["is_active"].agg(["sum", "count"]).rename(
    columns={"sum": "active", "count": "total"}
)


### 2.3 Daily PM2.5 measurements per city

In [ ]:
measurements = {}
for city in CITIES:
    path = f"{DATA_DIR}/{city}_measurements.json"
    if os.path.exists(path):
        with open(path) as f:
            measurements[city] = json.load(f)
        total_days = sum(len(s["daily_averages"]) for s in measurements[city])
        print(f"{city}: {len(measurements[city])} active sensor(s), {total_days} total daily records")
    else:
        print(f"{city}: no measurements file found")


### 2.4 Pollutant monitoring frequency (Q2 data source)

In [ ]:
pollutant_freq = pd.read_csv(f"{DATA_DIR}/pollutant_frequency.csv")
pollutant_freq[pollutant_freq["station_count"] > 0].sort_values(
    ["city", "station_count"], ascending=[True, False]
)


## 3. Data Quality Assessment

**Owner:** Paul

**TODO:** Missing values, duplicate records, outliers, completeness, and consistency checks. Use `sensors_df` and `measurements` from section 2. Already-known findings to formalize here: dead sensors (e.g. Nairobi station last reporting 2018), low-completeness active stations (e.g. sensors with 1-13 daily records), and the Johannesburg single-station limitation.

In [ ]:
# TODO (Paul): 3. Data Quality Assessment


## 4. Cleaning & Preprocessing

**Owner:** Paul (with Samuel reviewing)

**TODO:** Handle missing values, de-duplicate, standardize timestamps/timezones/units, merge per-city measurement data into one master DataFrame with a `city` column. Document every decision made. Output: data/cleaned/air_quality_master.csv

In [ ]:
# TODO (Paul (with Samuel reviewing)): 4. Cleaning & Preprocessing


## 5. Exploratory Data Analysis & Statistical Techniques

**Owner:** Winnie

**TODO:** Apply at least 3 statistical techniques answering Q1, Q3, Q4, Q5: descriptive/ranking (highest avg PM2.5), trend/seasonal analysis, WHO exceedance-rate analysis, completeness scoring.

In [ ]:
# TODO (Winnie): 5. Exploratory Data Analysis & Statistical Techniques


## 6. Visualizations

**Owner:** Gloria

**TODO:** At least 5 charts, each with a one-line justification for the chart type chosen. Suggested: city PM2.5 ranking, pollutant frequency by city, seasonal variation, WHO exceedance over time, station completeness, station location map.

In [ ]:
# TODO (Gloria): 6. Visualizations


## 7. Findings & Recommendations

**Owner:** Gloria (with input from all)

**TODO:** Evidence-backed answers to all 6 client questions, each citing the specific stat/chart that supports it. Prioritization recommendation for UNEP: which cities need intervention, and why.

In [ ]:
# TODO (Gloria (with input from all)): 7. Findings & Recommendations
